# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [3]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [4]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
openai = OpenAI (
    api_key=os.getenv('OLLAMA_API_KEY'),
    base_url=os.getenv('OLLAMA_BASE_URL')
)

In [6]:
# Some lists!

checklist = []
completed = []

In [7]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [8]:
get_checklist_report()

''

In [9]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [11]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [13]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [16]:
tools = [
    {
        "type": "function", 
        "function": create_checklist_json
    },
    {
        "type": "function", 
        "function": mark_complete_json
    }
]

In [17]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [18]:
choose_model_0="qwen2.5:7b"
choose_model_1="llama3.2:latest"
choose_model_3="mapler/gpt2:latest"
choose_model_2="gemma3:270m"

In [19]:
def loop(messages):
    response = openai.chat.completions.create(model=choose_model_0, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=choose_model_0, messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
checklist, completed = [], []
loop(messages)

Checklist #1: Define the speed of each train
Checklist #2: Determine how far the first train has traveled by the time the second starts
Checklist #3: Calculate the combined speed at which the distance between the trains decreases
Checklist #4: Formulate the equation representing the distance between the trains
Checklist #5: Solve the equation to find the time at which the trains meet

The first train travels at 60 mph, and the second at 80 mph.

Checklist #1: Define the speed of each train
Checklist #2: Determine how far the first train has traveled by the time the second starts
Checklist #3: Calculate the combined speed at which the distance between the trains decreases
Checklist #4: Formulate the equation representing the distance between the trains
Checklist #5: Solve the equation to find the time at which the trains meet

By the time the second train starts at 3:00 pm, the first train has already traveled for 1 hour, covering a 
distance of 60 miles.

Checklist #1: Define the speed of each train
Checklist #2: Determine how far the first train has traveled by the time the second starts
Checklist #3: Calculate the combined speed at which the distance between the trains decreases
Checklist #4: Formulate the equation representing the distance between the trains
Checklist #5: Solve the equation to find the time at which the trains meet

...

The equation representing the distance between the trains is 60 mi + 60 mph * t - 80 mph * t = 0, where t is the 
time in hours after 3:00 pm.

Checklist #1: Define the speed of each train
Checklist #2: Determine how far the first train has traveled by the time the second starts
Checklist #3: Calculate the combined speed at which the distance between the trains decreases
Checklist #4: Formulate the equation representing the distance between the trains
Checklist #5: Solve the equation to find the time at which the trains meet

60 + 60t - 80t = 0, simplifying we get 60 = 20t, giving t = 3 hours.

Checklist #1: Define the speed of each train
Checklist #2: Determine how far the first train has traveled by the time the second starts
Checklist #3: Calculate the combined speed at which the distance between the trains decreases
Checklist #4: Formulate the equation representing the distance between the trains
Checklist #5: Solve the equation to find the time at which the trains meet

All steps are marked as completed. Let's proceed to solve the problem step by step.

1. Define the speed of each train:
   - Train 1 (Boston-bound): 60 mph
   - Train 2 (New York-bound): 80 mph

2. Determine how far the first train has traveled by the time the second starts:
   - Train 1 starts at 2:00 pm and the second train starts at 3:00 pm, so Train 1 has traveled for 1 hour by the 
time the second train starts.
   - Distance traveled by Train 1 in 1 hour = 60 miles

3. Calculate the combined speed at which the distance between the trains decreases:
   - Combined speed = 60 mph (Train 1) + 80 mph (Train 2) = 140 mph

4. Formulate the equation representing the distance between the trains:
   - Let \( t \) be the time in hours after 3:00 pm.
   - Distance between the trains at 3:00 pm = 60 miles (initial distance) - 140 \(\cdot t\) (distance closing)
   - The distance between the trains will be 0 when the trains meet, so the equation is:
   - \( 60 - 140t = 0 \)

5. Solve the equation to find the time at which the trains meet:
   [ 60 - 140t = 0 \]
   [ -140t = -60 \]
   [ t = \frac{60}{140} \]
   [ t = \frac{3}{7} \text{ hours} \]

To convert \(\frac{3}{7}\) hours into minutes:
[ \frac{3}{7} \times 60 \approx 25.71 \text{ minutes} \]

Since 3:00 pm + 25.71 minutes is approximately 3:26 pm, the trains will meet around 3:26 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>